# 03 — Monitoring Model di Produksi dan Data Drift

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/13-mlops/03_monitoring_dan_drift.ipynb)

Model yang bagus hari ini bisa memburuk bulan depan tanpa satu baris kode pun berubah, karena **dunia berubah**: gaya bahasa pengguna, produk baru, musim, pengguna dari daerah lain. Perubahan distribusi data ini disebut **drift**:

| Jenis | Yang berubah | Contoh |
|---|---|---|
| **Data drift** (*covariate shift*) | Distribusi input `P(x)` | Pengguna mulai menulis dalam bahasa Jawa |
| **Label drift** (*prior shift*) | Distribusi label `P(y)` | Setelah gangguan layanan, ulasan negatif melonjak |
| **Concept drift** | Hubungan input → label `P(y\|x)` | Kata "sakit" yang dulu negatif kini jadi slang positif |

Masalah utamanya: di produksi, **label asli biasanya datang terlambat** (atau tidak pernah). Kita tidak bisa langsung menghitung akurasi, jadi yang dipantau adalah sinyal yang tersedia seketika: statistik input, distribusi prediksi, dan keyakinan model.

Isi notebook:
1. Simulasi 6 minggu trafik produksi dengan drift yang realistis: NusaX adalah dataset **paralel**, jadi kalimat test yang sama tersedia dalam bahasa Indonesia, Jawa, Sunda, dan Inggris.
2. Metrik monitoring: KS test, PSI, rasio token tak dikenal, keyakinan.
3. Laporan dan peringatan per minggu.
4. Tindak lanjut: **latih ulang** (*challenger*), bandingkan dengan *champion*, dan promosikan lewat model registry MLflow.

## 0. Instalasi, Kode, dan Model Champion

In [ ]:
%pip install -q mlflow torch pandas onnx onnxscript onnxruntime scipy matplotlib

Modul ini memakai kembali model sentimen dari modul 12 ([`12-deployment/app/`](../12-deployment/app/)): `model.py` (tokenizer, model, `Prediktor`) dan `latih.py` (training). Di Colab, file-file itu diunduh otomatis dari GitHub.

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"          # sembunyikan pesan petunjuk MLflow
RAW = "https://raw.githubusercontent.com/riki-profile/ai-engineer-journey/main"

APP12 = Path("../12-deployment/app")                   # jika repo di-clone: pakai file lokal
if not (APP12 / "model.py").exists():                  # di Colab: unduh dari GitHub
    APP12 = Path("app12")
    APP12.mkdir(exist_ok=True)
    for f in ["model.py", "latih.py"]:
        urllib.request.urlretrieve(f"{RAW}/12-deployment/app/{f}", APP12 / f)
if not Path("pyfunc_sentimen.py").exists():
    urllib.request.urlretrieve(f"{RAW}/13-mlops/pyfunc_sentimen.py", "pyfunc_sentimen.py")
sys.path.insert(0, str(APP12.resolve()))

EPOCH = 3 if QUICK_RUN else 30
print(f"Kode modul 12 dari: {APP12.resolve()} | QUICK_RUN: {QUICK_RUN} | epoch: {EPOCH}")

Model yang sedang "berjalan di produksi" (*champion*): dilatih hanya dengan data bahasa Indonesia, seperti di modul 12.

In [ ]:
import numpy as np
import pandas as pd

from latih import LABEL_ID, latih, muat_data
from model import LABELS, Prediktor, tokenisasi

config_champion = latih("artefak_champion", epoch=EPOCH, seed=0)
champion = Prediktor("artefak_champion")
print({k: config_champion[k] for k in ["bahasa", "akurasi_val", "akurasi_test"]})

## 1. Data Referensi dan Simulasi Trafik Produksi

- **Referensi**: data yang "dikenal" model, yaitu data train + validasi bahasa Indonesia. Statistik produksi dibandingkan dengan referensi ini.
- **Produksi**: 6 minggu trafik simulasi:

| Minggu | Isi | Yang disimulasikan |
|---|---|---|
| 1–2 | Kalimat test bahasa Indonesia | Normal (tanpa drift) |
| 3 | Bahasa Indonesia, mayoritas negatif | Label drift (misalnya setelah gangguan layanan) |
| 4 | Bahasa Jawa | Data drift (pengguna baru dari Jawa) |
| 5 | Bahasa Sunda | Data drift |
| 6 | Bahasa Inggris | Data drift berat |

Di dunia nyata, data produksi berasal dari log permintaan API (modul 12). Simpan **fitur/statistik**, bukan teks mentah, jika teksnya mengandung data pribadi.

In [ ]:
N = 80 if QUICK_RUN else 200
ID_KE_LABEL = {v: LABELS[v] for v in range(3)}

x_train, y_train = muat_data("train")
x_val, y_val = muat_data("valid")
referensi = pd.DataFrame({"teks": x_train + x_val, "label": np.concatenate([y_train, y_val])})

test = {b: pd.DataFrame(dict(zip(["teks", "label"], muat_data("test", b))))
        for b in ["indonesian", "javanese", "sundanese", "english"]}
rng = np.random.default_rng(0)
negatif = test["indonesian"][test["indonesian"]["label"] == LABEL_ID["negative"]]
lainnya = test["indonesian"][test["indonesian"]["label"] != LABEL_ID["negative"]]
minggu3 = pd.concat([negatif.sample(min(len(negatif), int(N * 0.8)), random_state=0),
                     lainnya.sample(int(N * 0.2), random_state=0)])

MINGGU = {
    "M1 indonesia": test["indonesian"].iloc[:N],
    "M2 indonesia": test["indonesian"].iloc[N:2 * N],
    "M3 banyak negatif": minggu3,
    "M4 jawa": test["javanese"].iloc[:N],
    "M5 sunda": test["sundanese"].iloc[:N],
    "M6 inggris": test["english"].iloc[:N],
}
for nama, df in MINGGU.items():
    print(f"{nama:18} {len(df):4} kalimat | contoh: {df['teks'].iloc[0][:70]}")

## 2. Metrik Monitoring

Untuk setiap minggu kita hitung sinyal yang **tidak butuh label**:

| Metrik | Mengukur | Cara |
|---|---|---|
| **Panjang teks** | Perubahan bentuk input | *Kolmogorov–Smirnov (KS) test*: apakah dua sampel angka berasal dari distribusi yang sama? p-value kecil = berbeda |
| **Rasio token tak dikenal (UNK)** | Kosakata baru yang tidak dikenal model | Porsi token yang tidak ada di vocab training |
| **Distribusi prediksi** | Perubahan output model | **PSI** (*Population Stability Index*) antara proporsi prediksi referensi vs minggu ini |
| **Keyakinan rata-rata** | Model makin ragu | Rata-rata probabilitas tertinggi |

Dan satu metrik yang **butuh label** (datang terlambat, misalnya dari anotasi sampel): **akurasi**.

**PSI** = Σ (p_minggu − p_ref) × ln(p_minggu / p_ref) atas semua kategori/bin. Aturan praktis yang umum dipakai: < 0,1 stabil, 0,1–0,25 perlu diperhatikan, > 0,25 berubah signifikan.

In [ ]:
from scipy.stats import ks_2samp


def psi(ref, kini, kategori):
    eps = 1e-4
    p_ref = np.array([np.mean(ref == k) for k in kategori]) + eps
    p_kini = np.array([np.mean(kini == k) for k in kategori]) + eps
    return float(np.sum((p_kini - p_ref) * np.log(p_kini / p_ref)))


def statistik(df, model):
    teks = df["teks"].tolist()
    prob = np.array([[h["skor"][lab] for lab in LABELS] for h in model.prediksi(teks)])
    token = [t for x in teks for t in tokenisasi(x) if "_" not in t]          # hanya kata tunggal
    return {"panjang": np.array([len(x.split()) for x in teks]),
            "unk": np.mean([t not in model.vocab.token_ke_id for t in token]),
            "prediksi": prob.argmax(1), "yakin": prob.max(1), "label": df["label"].to_numpy()}


ref = statistik(referensi, champion)


def laporan(model, ref):
    baris = []
    for nama, df in MINGGU.items():
        s = statistik(df, model)
        baris.append({"minggu": nama,
                      "KS p-value panjang": ks_2samp(ref["panjang"], s["panjang"]).pvalue,
                      "rasio UNK": s["unk"],
                      "PSI prediksi": psi(ref["prediksi"], s["prediksi"], range(3)),
                      "keyakinan": s["yakin"].mean(),
                      "% negatif diprediksi": np.mean(s["prediksi"] == 0),
                      "akurasi (label terlambat)": np.mean(s["prediksi"] == s["label"])})
    return pd.DataFrame(baris).set_index("minggu")


tabel = laporan(champion, ref)
print(f"Referensi: rasio UNK {ref['unk']:.3f} | keyakinan {ref['yakin'].mean():.3f}")
tabel.round(3)

Catatan: rasio UNK referensi dihitung pada data train+validasi, yaitu data yang dipakai membangun vocab, sehingga lebih rendah daripada data baru. Rasionya tidak nol karena kata yang muncul kurang dari 2 kali tidak masuk vocab. Karena itu, nilai "normal" di produksi diambil dari minggu 1–2 (data baru berbahasa sama).

## 3. Peringatan dan Dashboard

Aturan peringatan sederhana (ambang disesuaikan dengan pengalaman di sistemmu):

In [ ]:
ATURAN = {
    "input berubah (KS p < 0,01)": lambda b: b["KS p-value panjang"] < 0.01,
    "kosakata asing (UNK > 2× normal)": lambda b: b["rasio UNK"] > 2 * tabel["rasio UNK"].iloc[:2].mean(),
    "prediksi bergeser (PSI > 0,25)": lambda b: b["PSI prediksi"] > 0.25,
    "prediksi mulai bergeser (PSI 0,1–0,25)": lambda b: 0.1 < b["PSI prediksi"] <= 0.25,
    "model ragu (keyakinan turun > 10%)": lambda b: b["keyakinan"] < 0.9 * tabel["keyakinan"].iloc[:2].mean(),
}
for nama, b in tabel.iterrows():
    pemicu = [aturan for aturan, cek in ATURAN.items() if cek(b)]
    print(f"{'⚠️ ' if pemicu else '✅'} {nama:18} {', '.join(pemicu) if pemicu else 'normal'}")

In [ ]:
import matplotlib.pyplot as plt

kolom = ["rasio UNK", "PSI prediksi", "keyakinan", "akurasi (label terlambat)"]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, k in zip(axes, kolom):
    ax.plot(range(1, len(tabel) + 1), tabel[k], "o-")
    ax.set(title=k, xticks=range(1, len(tabel) + 1), xlabel="minggu")
axes[1].axhline(0.25, color="red", ls="--", lw=1)
plt.tight_layout()
plt.show()

Cara membaca hasilnya:
- **Minggu 3 (label drift)**: input masih bahasa Indonesia (UNK normal), tetapi **distribusi prediksi** bergeser ke negatif: PSI naik dan porsi prediksi negatif meningkat. Ini bisa berarti masalah bisnis sungguhan (pelanggan kecewa), bukan masalah model. Monitoring model juga berfungsi sebagai monitoring bisnis.
- **Minggu 4–6 (data drift)**: **rasio UNK** melonjak karena kosakata bahasa daerah/Inggris tidak dikenal model, dan akurasi (saat label datang) biasanya turun. Rasio UNK adalah sinyal dini yang sangat berguna untuk model teks.
- Drift **tidak selalu** menurunkan performa, dan performa bisa turun tanpa drift yang terlihat (concept drift). Karena itu tetap kumpulkan label untuk sebagian sampel produksi (misalnya anotasi manusia untuk 1% data).

## 4. Tindak Lanjut: Challenger vs Champion

Drift bahasa daerah terdeteksi. Tindakan yang masuk akal: latih **model penantang** (*challenger*) dengan data yang mencakup bahasa baru, lalu bandingkan dengan *champion* **sebelum** mengganti model produksi. Keputusan promosi harus berbasis aturan yang jelas, misalnya: lebih baik di trafik campuran **dan** tidak lebih buruk di bahasa Indonesia (bahasa utama).

In [ ]:
config_challenger = latih("artefak_challenger", epoch=EPOCH, seed=0, bahasa=("indonesian", "javanese", "sundanese"))
challenger = Prediktor("artefak_challenger")

baris = []
for bahasa, df in test.items():
    for nama, model in [("champion", champion), ("challenger", challenger)]:
        pred = np.array([LABELS.index(h["label"]) for h in model.prediksi(df["teks"].tolist())])
        baris.append({"bahasa": bahasa, "model": nama, "akurasi": np.mean(pred == df["label"].to_numpy())})
perbandingan = pd.DataFrame(baris).pivot(index="bahasa", columns="model", values="akurasi")
perbandingan.loc["rata-rata jawa+sunda"] = perbandingan.loc[["javanese", "sundanese"]].mean()
perbandingan.round(3)

Perhatikan bahwa bahasa Jawa dan Sunda di sini memakai **test set** NusaX, sedangkan challenger dilatih dengan **train set** bahasa tersebut. Karena NusaX paralel, kalimat test bahasa Jawa adalah terjemahan kalimat test bahasa Indonesia, bukan kalimat training, sehingga perbandingan tetap adil. Bahasa Inggris tidak ikut training challenger, dan hasilnya menunjukkan apakah model menggeneralisasi ke bahasa yang belum pernah dilihat.

Terakhir, catat keduanya di **model registry** MLflow (notebook 01) dan promosikan jika lolos aturan:

In [ ]:
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri(f"sqlite:///{Path('mlflow.db').resolve()}")
mlflow.set_experiment("sentimen-monitoring")
client = MlflowClient()
NAMA_MODEL = "sentimen-id-monitoring"


def daftarkan(folder, config, alias):
    with mlflow.start_run(run_name=alias):
        mlflow.log_params({k: str(config[k]) for k in ["dim", "lr", "epoch", "seed", "bahasa"]})
        mlflow.log_metrics({f"akurasi_{b}": perbandingan.loc[b, alias] for b in test})
        info = mlflow.pyfunc.log_model(name="model", python_model="pyfunc_sentimen.py", artifacts={"artefak": folder},
                                       code_paths=[str(APP12 / "model.py")], registered_model_name=NAMA_MODEL,
                                       input_example=pd.DataFrame({"teks": ["Barangnya bagus sekali"]}),
                                       pip_requirements=["onnxruntime", "numpy", "pandas"])
    client.set_registered_model_alias(NAMA_MODEL, alias, info.registered_model_version)
    return info.registered_model_version


v_champion = daftarkan("artefak_champion", config_champion, "champion")
v_challenger = daftarkan("artefak_challenger", config_challenger, "challenger")

lebih_baik_campuran = perbandingan.loc["rata-rata jawa+sunda", "challenger"] > perbandingan.loc["rata-rata jawa+sunda", "champion"]
tidak_merusak_indonesia = perbandingan.loc["indonesian", "challenger"] >= perbandingan.loc["indonesian", "champion"] - 0.02
if lebih_baik_campuran and tidak_merusak_indonesia:
    client.set_registered_model_alias(NAMA_MODEL, "champion", v_challenger)
    print(f"✅ Challenger (versi {v_challenger}) dipromosikan menjadi champion")
else:
    print(f"❌ Challenger tidak dipromosikan: lebih baik di jawa+sunda={lebih_baik_campuran}, "
          f"bahasa Indonesia tetap baik={tidak_merusak_indonesia}")
print("Alias sekarang:", client.get_registered_model(NAMA_MODEL).aliases)

Di produksi, promosi biasanya dilakukan bertahap: **shadow deployment** (challenger ikut memprediksi tetapi hasilnya tidak dipakai, hanya dibandingkan), lalu **canary** (sebagian kecil trafik, misalnya 5%, dilayani challenger), baru kemudian semua trafik. Jika ada masalah, **rollback** cukup dengan memindahkan alias kembali.

**Alat monitoring:** *Evidently* (laporan drift siap pakai, open source), *NannyML* (memperkirakan performa tanpa label), atau *Prometheus + Grafana* untuk metrik operasional (latensi, error rate dari middleware modul 12). Platform cloud (SageMaker Model Monitor, Vertex AI Model Monitoring) menyediakan versi terkelola.

## Ringkasan

- **Drift**: data drift `P(x)`, label drift `P(y)`, concept drift `P(y|x)`. Label produksi datang terlambat, jadi pantau sinyal tanpa label.
- Sinyal utama: **KS test** untuk fitur numerik, **PSI** untuk distribusi prediksi/kategori, **rasio UNK** untuk model teks, **keyakinan** rata-rata; lalu **akurasi** saat label tersedia.
- Buat **aturan peringatan** yang jelas dan dashboard per periode; ingat bahwa drift juga bisa menandakan perubahan bisnis.
- Tindak lanjut: latih **challenger**, bandingkan dengan **champion** memakai aturan promosi yang eksplisit, promosikan lewat **alias** registry; gunakan shadow/canary dan siapkan rollback.
- Ini menutup siklus MLOps: data → training → registrasi → deployment → monitoring → training ulang.

## Latihan

1. **PSI untuk fitur numerik.** Implementasikan PSI untuk panjang teks dengan bin berdasarkan kuantil data referensi (misalnya 10 bin) dan tambahkan ke laporan. Bandingkan kepekaannya dengan KS p-value, terutama saat jumlah sampel per minggu kecil (coba N=30) dan besar (N=400).
2. **Drift bertahap.** Buat 10 minggu di mana porsi kalimat bahasa Jawa naik 10% setiap minggu (sisanya bahasa Indonesia). Di minggu ke berapa setiap aturan peringatan pertama kali menyala? Aturan mana yang paling cepat mendeteksi drift?
3. **Monitoring dari API.** Ubah middleware API modul 12 agar mencatat ke file JSONL, untuk setiap permintaan: waktu, panjang teks, rasio UNK, label prediksi, dan keyakinan, **tanpa** teks mentahnya. Kirim 300 permintaan campuran (Indonesia + Jawa), lalu buat laporan monitoring seperti di atas dari file log tersebut.